# Notebook 03
Estes gráficos foram construídos para testar duas hipóteses sobre o mercado editorial e seus leitores:
- O formato dita o tamanho do livro? (Gráfico KDE): Capa Dura é usado apenas para livros grandes? E-books tendem a ser mais curtos?
- O gênero dita a forma de consumo? (Mix por Gênero): Os leitores estão indo para o meio digital ou preferindo edições físicas.

> ### ⚠️ Delimitação do Estudo
> Esta análise baseia-se **exclusivamente no conjunto de dados retirados do Kaggle**. Os padrões e associações mapeados refletem o comportamento histórico desta base específica e não devem ser generalizados como uma regra universal para todo o mercado editorial global.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import combinations
from collections import Counter
from scipy.stats import gaussian_kde

# reprodutibilidade
np.random.seed(42)

# estilo base
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.autolayout': True})
PALETTE = [
    '#6e4e48', '#0071bc', '#13939a', '#ff7363', '#f9a27b', '#ffc364'
]
PALETTE_FILL = [
    'rgba(110, 78, 72, 0.15)',
    'rgba(0, 113, 188, 0.15)',
    'rgba(19, 147, 154, 0.15)',
    'rgba(255, 115, 99, 0.15)',
    'rgba(249, 162, 123, 0.15)',
    'rgba(255, 195, 100, 0.15)'
]

# ── Thresholds globais ────────────────────────────────────────────────────────
PARETO_CUTOFF       = 0.80   # percentual para o corte de Pareto
KDE_FORMAT_MIN_N    = 30 
MIN_SUPPORT_PCT   = 0.01
BAYESIAN_QUANTILE   = 0.75 


# tags guarda-chuvas
STOP_GENRES = {
    'Fiction', 'Literature', 'Books',
    'Audiobook', 'Audiobooks', 'Nonfiction'
}

print('Configuração carregada.')

Configuração carregada.


## 2. Ingestão e Limpeza

Leitura do parquet, coerção de tipos numéricos, remoção de linhas sem dados essenciais e deduplicação por ISBN.  
O volume após cada filtro é registrado para rastreabilidade do pipeline.

In [ ]:
df = pd.read_parquet('datasets/books.parquet')
print(f'Leitura bruta:          {len(df):>8,} linhas')

# deduplicação
df = df.dropna(subset=['isbn'])
df = df.drop_duplicates(subset=['isbn'])
print(f'Após dedup por ISBN:    {len(df):>8,} linhas')

# coerção de tipos numéricos
num_cols = ['pages', 'rating', 'reviews', 'totalratings']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# remoção de linhas sem dados essenciais
df = df.dropna(subset=['rating', 'totalratings', 'genre'])

# Frequência mínima para o Lift (1% do total de livros válidos)
LIFT_FREQ_MIN = int(len(df) * MIN_SUPPORT_PCT)
Q1_pages  = df['pages'].quantile(0.25)
Q3_pages  = df['pages'].quantile(0.75)
IQR_pages = Q3_pages - Q1_pages
PAGES_MIN = max(1, int(Q1_pages - 1.5 * IQR_pages))
PAGES_MAX = int(df['pages'].quantile(0.99))

# ── Resumo das variáveis geradas
print("⚙️ Parâmetros calculados dinamicamente para este acervo:")
print(f"Total de livros (df): {len(df):,}")
print(f"Frequência mínima para Lift: {LIFT_FREQ_MIN} ocorrências")
print(f"Corte inferior de páginas:   {PAGES_MIN} páginas")
print(f"Corte superior de páginas:   {PAGES_MAX} páginas")


Leitura bruta:            84,054 linhas
Após dedup por ISBN:      73,415 linhas
⚙️ Parâmetros calculados dinamicamente para este acervo:
Total de livros (df): 73,415
Frequência mínima para Lift: 734 ocorrências
Corte inferior de páginas:   1 páginas
Corte superior de páginas:   892 páginas


## 3. Engenharia de Variáveis

Transforma os dados brutos em métricas analíticas e prepara a taxonomia de gêneros.

- **`engagement_ratio`:** proporção de leitores que escreveram uma resenha textual em relação aos que apenas avaliaram com estrelas — proxy de comunidade ativa.
- **`bayesian_rating`:** nota ajustada pelo volume de avaliações via prior bayesiano, eliminando o viés de livros com nota máxima em pouquíssimos votos.
- **`genre_list`:** lista de gêneros válidos por livro, sem tags guarda-chuva definidas em `STOP_GENRES`.
- **`genre_count`:** quantidade de gêneros válidos — variável central da análise de hibridismo.
- **`log_pages`:** escala logarítmica de páginas para uso nos gráficos de KDE.

In [ ]:
# engagement_ratio
df['engagement_ratio'] = np.where(
    df['totalratings'] > 0,
    df['reviews'] / df['totalratings'],
    0
)

# bayesian_rating — prior = média global, peso = BAYESIAN_QUANTILE
C = df['rating'].mean()
m = df['totalratings'].quantile(BAYESIAN_QUANTILE)
df['bayesian_rating'] = (
    (df['totalratings'] / (df['totalratings'] + m)) * df['rating'] +
    (m              / (df['totalratings'] + m)) * C
)

# genre_list — parse, limpeza de stop_genres, dedup interno por livro
df['genre'] = df['genre'].fillna('')
df['genre_list'] = (
    df['genre']
    .apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])
    .apply(lambda x: list(dict.fromkeys(g for g in x if g not in STOP_GENRES)))
)
df['genre_count'] = df['genre_list'].apply(len)

# log_pages — apenas para livros dentro dos limites definidos na configuração
df['log_pages'] = np.where(
    df['pages'].between(PAGES_MIN, PAGES_MAX),
    np.log10(df['pages']),
    np.nan
)

print(f'Após feature engineering: {len(df):>8,} linhas')
print()
print(df[['engagement_ratio', 'bayesian_rating', 'genre_count']].describe().round(3).to_string())

Após feature engineering:   73,415 linhas

       engagement_ratio  bayesian_rating  genre_count
count         73415.000        73415.000    73415.000
mean              0.111            3.909        6.766
std               0.083            0.117        3.390
min               0.000            2.534        0.000
25%               0.057            3.875        4.000
50%               0.097            3.900        7.000
75%               0.148            3.935       10.000
max               2.000            4.733       16.000


C:\Users\Clara\AppData\Roaming\Python\Python314\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


## 4. Redução por Pareto — definição de `top_genres`

Aplicamos o Princípio de Pareto para isolar as tags de gênero com real relevância volumétrica.  
O corte é determinado dinamicamente pelo `PARETO_CUTOFF`.

O conjunto `top_genres` resultante é reutilizado como filtro em todas as etapas seguintes.

In [ ]:
# frequência de cada gênero no acervo
all_genres = [g for sublist in df['genre_list'] for g in sublist]
genre_counts = Counter(all_genres)

pareto_df = (
    pd.DataFrame.from_dict(genre_counts, orient='index', columns=['Frequencia'])
    .sort_values('Frequencia', ascending=False)
    .reset_index(names='Genero')
)
pareto_df['Pct_Cumulativa'] = (
    pareto_df['Frequencia'].cumsum() / pareto_df['Frequencia'].sum()
)

# corte dinâmico pelo PARETO_CUTOFF
pareto_cut = pareto_df[pareto_df['Pct_Cumulativa'] <= PARETO_CUTOFF]
top_genres = set(pareto_cut['Genero'])

total_tags = len(pareto_df)
top_n      = len(top_genres)
print(f'Total de tags únicas:  {total_tags:>5}')
print(f'Tags no corte Pareto:  {top_n:>5}  ({top_n/total_tags*100:.1f}% das tags = {PARETO_CUTOFF*100:.0f}% do volume)')

# Gráfico de Pareto
plot_df = pareto_cut.copy()
plot_df['Pct_Cumulativa_100'] = plot_df['Pct_Cumulativa'] * 100

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=plot_df['Genero'],
        y=plot_df['Frequencia'],
        name='Frequência',
        marker_color='#2980B9'
    ),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=plot_df['Genero'],
        y=plot_df['Pct_Cumulativa_100'],
        name='% Cumulativa',
        mode='lines+markers',
        line=dict(color='#F39C12', width=3),
        marker=dict(size=4)
    ),
    secondary_y=True
)
fig.add_hline(
    y=PARETO_CUTOFF * 100,
    line_dash='dash', line_color='#7F8C8D', line_width=2,
    annotation_text=f'Corte de Pareto ({PARETO_CUTOFF*100:.0f}%)',
    annotation_position='top left',
    annotation_font_color='#7F8C8D',
    secondary_y=True
)
fig.update_layout(
    title_text='<b>Gráfico de Pareto: os gêneros que sustentam o acervo</b>',
    title_font_size=16,
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    height=600
)
fig.update_xaxes(title_text='Gêneros literários', tickangle=45, nticks=40)
fig.update_yaxes(title_text='Frequência absoluta (contagem de tags)', secondary_y=False)
fig.update_yaxes(title_text='Porcentagem cumulativa (%)', range=[0, 105], secondary_y=True)
fig.show()

Total de tags únicas:   1158
Tags no corte Pareto:    205  (17.7% das tags = 80% do volume)


## 5. Distribuição de páginas por formato — KDE normalizado

Compara a distribuição de páginas entre os formatos de livro usando **KDE (Kernel Density Estimation) normalizado** — cada curva tem área = 1, eliminando o viés de volume entre formatos com populações muito diferentes.

O eixo X está em escala logarítmica para comprimir outliers e revelar a forma real de cada distribuição.  
O eixo Y de densidade é suprimido intencionalmente: o valor absoluto não tem interpretação direta — o que importa é a forma e a posição de cada curva.

In [ ]:
PALETTE_PLOTLY = px.colors.qualitative.Prism
KDE_FORMAT_MIN_N = 50
df_kde = df[df['log_pages'].notna() & df['bookformat'].notna()].copy()
fmt_counts = df_kde['bookformat'].value_counts()
valid_fmts = fmt_counts[fmt_counts >= KDE_FORMAT_MIN_N].index.tolist()
df_kde = df_kde[df_kde['bookformat'].isin(valid_fmts)]


# KDE por grupo 
x_range  = np.linspace(df_kde['log_pages'].min(), df_kde['log_pages'].max(), 500)
tickvals = [50, 100, 200, 300, 500, 1000, 2000]
fig = go.Figure()


for i, fmt in enumerate(valid_fmts):
    subset = df_kde[df_kde['bookformat'] == fmt]['log_pages'].dropna()


    if len(subset) > 1:
        kde = gaussian_kde(subset, bw_method='scott')
        y   = kde(x_range)

        color_line = PALETTE_PLOTLY[i % len(PALETTE_PLOTLY)]

        if color_line.startswith('rgb'):
            color_fill = color_line.replace('rgb', 'rgba').replace(')', ', 0.15)')
        else: 
            color_fill = px.colors.label_rgb(px.colors.hex_to_rgb(color_line) + (0.15,))

        n = len(subset)

        fig.add_trace(go.Scatter(
            x=10**x_range,
            y=y,
            mode='lines',
            fill='tozeroy',
            fillcolor=color_fill,
            line=dict(color=color_line, width=2),
            name=f'{fmt}  (n={n:,})',
            opacity=0.9
        ))


fig.update_layout(
    title='<b>Distribuição de páginas por formato — densidade relativa (KDE)</b>',
    title_font_size=16,
    xaxis=dict(type='log', tickvals=tickvals, ticktext=[str(v) for v in tickvals], title='Número de páginas (escala log)'),
    yaxis=dict(title='Densidade relativa (área = 1 por formato)', showticklabels=False),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(title='Formato'),
    height=500
)

fig.show() 

## 6. Mix de bookformat por gênero — stacked bar 100%

Cada barra representa um gênero do `top_genres` (corte de Pareto) e soma **100%**, eliminando o viés de volume entre gêneros com populações muito diferentes.

O gênero é obtido via `explode` da `genre_list` — cada livro contribui uma linha por gênero, preservando os híbridos.  
As barras são ordenadas pelo share do formato dominante global, revelando padrões estruturais de consumo.

In [ ]:
# Pega os 20 gêneros mais frequentes diretamente do pareto
top20_genres = pareto_df.head(20)['Genero'].tolist()

# Prepara os dados ignorando livros sem formato ou sem gênero
df_fmt = df[
    df['bookformat'].notna() &
    df['genre_list'].apply(lambda x: len(x) > 0)
].copy()

# Explode a lista (uma linha por combinação de livro e gênero)
df_exploded = df_fmt.explode('genre_list')

# Mantém apenas os gêneros que estão no nosso Top 20
df_exploded = df_exploded[df_exploded['genre_list'].isin(top20_genres)]

print(f'Linhas após explode (Top 20 gêneros): {len(df_exploded):,}')
print(f'Gêneros únicos presentes:             {df_exploded["genre_list"].nunique()}')

# Tabela de contingência normalizada por gênero (0 a 100%)
pivot = (
    df_exploded
    .groupby(['genre_list', 'bookformat'])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
dominant_fmt = pivot_pct.sum(axis=0).idxmax()
pivot_pct    = pivot_pct.sort_values(dominant_fmt, ascending=True)

formats = pivot_pct.columns.tolist()
genres  = pivot_pct.index.tolist()

# ── Gráfico ───────────────────────────────────────────────────────────────────
PALETTE_PLOTLY = px.colors.qualitative.Prism

fig = go.Figure()

for i, fmt in enumerate(formats):
    fig.add_trace(go.Bar(
        name=fmt,
        x=pivot_pct[fmt].values,
        y=genres,
        orientation='h',
        marker_color=PALETTE_PLOTLY[i % len(PALETTE_PLOTLY)],
        hovertemplate='%{y}<br>' + fmt + ': %{x:.1f}%<extra></extra>'
    ))

fig.update_layout(
    barmode='stack',
    title=f'<b>Mix de formato por gênero — Top 20 maiores gêneros</b>',
    title_font_size=16,
    xaxis=dict(
        title='Proporção (%)',
        range=[0, 100],
        ticksuffix='%'
    ),
    yaxis=dict(
        title='Gênero literário',
        tickfont=dict(size=11),
        automargin=True
    ),
    legend=dict(title='Formato', traceorder='normal'),
    template='plotly_white',
    height=600,
    margin=dict(l=180)
)
fig.show()

Linhas após explode (Top 20 gêneros): 142,015
Gêneros únicos presentes:             20


## 8. Conclusão — respondendo à pergunta de pesquisa

* **O formato não define o tamanho da obra:** As publicações padrão *Paperback* e *Hardcover* concentram-se na mesma faixa das publicações *Ebook* aproximadamente na faixa de 300 páginas .
* **A hegemonia absoluta da Brochura:** O *Paperback* é o líder incontestável do nosso dataset, dominando 18 dos 20 maiores gêneros.
* **As exceções revelam o comportamento do leitor:** A regra da brochura só é quebrada por *Childrens* (54.6%) e *Biography* (48.6%), que têm o *Hardcover* como formato principal. Uma probabilidade disto acontecer seria que as capas duras são priorizadas onde há necessidade de durabilidade (livros infantis) ou apelo de prestígio e presente (biografias).